<p><small><small>This Notebook is made available subject to the licence and terms set out in the <a href = "http://www.github.com/google-deepmind/ai-foundations">AI Research Foundations Github README file</a>.</small></small></p>

<img src="https://storage.googleapis.com/dm-educational/assets/ai_foundations/GDM-Labs-banner-image-C7-white-bg.png">

# Lab: Train and Evaluate your Text Summarization Model

<a href='https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_8/gdm_lab_8_6_summarization_part_2.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

3-4 hours of activities and 1-3 hours of waiting for models to train

In this second lab, you will train and evaluate your text summarization model.

Similarly, as in the previous activities, this lab also contains a worked example for every step. Consider the code for the worked example as a template for your own implementation. Given that the steps for fine-tuning a summarization model do not vary too much, you may be able to run a lot of the existing code without changing it too much. However, some steps may still not be applicable to your project, and likewise, you may have to complete additional steps, so treat the template in this lab just as a starting point and feel free to add or delete code cells to this notebook as you see fit.


## Overview

In this second part of implementing your capstone project, you will focus on fine-tuning and evaluating your text summarization model.

### What you will learn

By the end of this second part of the capstone project, you will be able to:

* Fine-tune a large language model using LoRA to be used as a text summarization system.

* Perform systematic manual evaluation of text summarization responses.

### Tasks

In this lab, you will:

* Load the preprocessed dataset that you prepared in the first part.

* Configure the Gemma model and LoRA hyperparameters.

* Execute the fine-tuning process.

* Test the fine-tuned model and run manual spot checks on unseen data.

* Iterate on the hyperparameter settings and fine-tuning process until you are satisfied with your model's behavior.

The end result of this lab will be a text summarization model that can respond to user queries as specified in your task description.

### What you will use

This lab builds on materials from several previous courses. If you would like to refresh your knowledge on any of the following topics, you can find the relevant materials in the following courses:

- **General knowledge on LLMs and transformers** (Courses 01 Build Your Own Small Language Model, 03 Design and Train Neural Networks, and 04 Discover the Transformer Architecture).

- **Fine-tuning models using LoRA** (Courses 05 Fine-tune Your Model and 07 Accelerate Your Model).  


**This lab needs to be run on GPU. Choose a T4 GPU.** See the section "How to use Google Colaboratory (Colab)" below for instructions on how to do this.

## How to use Google Colaboratory (Colab)

Google Colaboratory (also known as Google Colab) is a platform that allows you to run Python code in your browser. The code is written in cells that are excuted on a remote server.

To run a cell, hover over a cell, and click on the `run` button to its left. The run button is the circle with the triangle (▶). Alternatively, you can also click on a cell and use the keyboard combination Ctrl+Return (or ⌘+Return if you are using a Mac).

To try this out, run the following cell. This should print today's day of the week below it.

In [ ]:
from datetime import datetime
print(f"Today is {datetime.today():%A}.")

Note that the order in which you run the cells matters. When you are working through a lab, make sure to always run all cells in order. Otherwise, the code might not work. If you take a break while working on a lab, Colab may disconnect you and in that case, you have to execute all cells again before  continuing your work. To make this easier, you can select the cell you are currently working on and then choose __Runtime → Run before__  from the menu above (or use the keyboard combination Ctrl/⌘ + F8). This will re-execute all cells before the current one.

### Using Colab with a GPU

Follow these steps to run the activities in this lab on a GPU:

1.  In the top menu bar, click on **Runtime**.
2.  Select **Change runtime type** from the dropdown menu.
3.  In the pop-up window under **Hardware Accelerator**, select **GPU** (usually listed as `T4 GPU`).
5.  Click **Save**.

Your Colab session will now restart with GPU access.

Note that access to GPUs is limited and at times, you may not be able to run this lab on a GPU. All activities will still work but they will run slower and you will have to wait longer for some of the cells to finish running.


## Imports

The following cell contains imports for a range of libraries that you may use for fine-tuning and evaluating your model. Depending on your implementation, you may need to import additional libraries.

In [ ]:
# Standard library imports.
import io # Input/output operations.
import json # JSON data handling.
import os # Operating system interface.
import random # For random number generation and shuffling of data.
import re # Regular expressions for text processing.
import unicodedata # Unicode character database.
from textwrap import fill # Text wrapping utilities.
from typing import Any, Dict, Tuple, List

# Third-party data and utility libraries.
import numpy as np # Numerical computing.
import pandas as pd # Data manipulation and analysis.
from tqdm import tqdm # Progress bars.

import matplotlib.pyplot as plt # Plotting and visualisation.

# Set Keras and JAX settings.
os.environ["KERAS_BACKEND"] = "jax"

# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

# Deep learning and AI libraries.
import jax.numpy as jnp # JAX numerical computing.
import keras # High-level neural networks API.
import keras_nlp # Keras NLP extensions.

# Google Colab specific.
from google.colab import drive # Access to Google Drive.
from google.colab import userdata # Access to Colab secrets.


##  Activity 1: Load your data

As a first step, you will load your training and test data that you prepared in the previous activity from Google Drive. See the worked example for how to do this.

In [ ]:
# Connect this notebook to Google Drive.
drive.mount('/content/drive')

#### Worked example: Load data


```python
train_df = pd.read_json(
    "/content/drive/MyDrive/summarization_training_data.jsonl", lines=True
)

test_df = pd.read_json(
    "/content/drive/MyDrive/summarization_test_data.jsonl", lines=True
)

# Turn dataframe into a list of dictionaries, as expected by the model
# fine-tuning code.
train_data = train_df.to_dict('records')
test_data = test_df.to_dict('records')

# Print the first element in train_data, val_data, test_data for verifcation.
print(train_data[0])
print(test_data[0])
```

#### Your implementation


In [ ]:
# Add your code for loading your data here.


### Configure your Kaggle API key for Gemma

To access Gemma models, provide your Kaggle credentials. If you have not set your Kaggle API key yet take a look at the instruction in the following cells.

#### Set up a Kaggle account



To run this notebook, you will have to sign up for [Kaggle](https://www.kaggle.com), a platform that hosts datasets and models for machine learning, and sign the agreement for using the Gemma 3 model. This is required so that you can download the weights of the Gemma model for fine-tuning.

**Step 1: Create your Kaggle account**

* Go to the Kaggle website: https://www.kaggle.com

* Click the "Register" button in the top-right corner.

* You can sign up using your Google account (recommended for easy Colab integration) or by entering an email and password.

* Follow the on-screen prompts to complete your registration and verify your email.

**Step 2: Sign the Gemma 3 model agreement**

* Make sure you are logged into your new Kaggle account.

* Go directly to the Gemma 3 model card page: https://www.kaggle.com/models/keras/gemma3/keras/

* You should see a "Request Access" button.

* Click the button, read through the license agreement, and click "Accept" to gain access to the model. You must do this before the API will let you download the model.

**Step 3: Generate your Kaggle API key**

* From any Kaggle page, click on your profile picture or icon in the top-right corner.

* Select "Account" from the drop-down menu.

* Scroll down to the "API" section.

* Click the "Create Legacy API Key" button in the "Legacy API Credentials" section.

* This will immediately download a file named `kaggle.json` to your computer. This file contains your username and your secret API key. Keep it safe.

**Step 4: Set your API Key in Colab**

* Click the "key" icon 🔑 in the left-hand sidebar.

* You will see the "Secrets" panel.

* Now, open the kaggle.json file you downloaded on your computer. It's a simple text file and will look like this:

   ```json
   {"username":"YOUR_KAGGLE_USERNAME","key":"YOUR_KAGGLE_API_KEY"}
   ```
* In the Colab Secrets panel, create two new secrets:

   1. Name: `KAGGLE_USERNAME`

      Value: Copy and paste `YOUR_KAGGLE_USERNAME` from your `kaggle.json` file.

   2. Name: `KAGGLE_KEY`

      Value: Copy and paste `YOUR_KAGGLE_API_KEY` from your `kaggle.json` file.

* For both secrets, make sure the "Notebook access" toggle is switched on.


#### Load the Kaggle username and key

In [ ]:
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

## Activity 2: Preparation for fine-tuning

Before starting fine-tuning, ensure that your training data is formatted correctly for the specific model you plan to use. In the previous notebook, you created JSONL files containing classification examples, which can be reused across different models and fine-tuning methods.

However, each model has its own formatting requirements. The Gemma models support only two conversational roles, **user** and **model**, but not a **system** role. In other words, prompts and responses are structured as direct exchanges between the user and the model, without a separate system message to provide global instructions or context (such as defining tone, style, or constraints). Higher-capacity LLMs, like Gemini, also use a system role for this purpose, allowing developers to set overall behavior at the start of a conversation. Because Gemma does not support the system role, each training example must explicitly include any necessary context or instruction within the user prompt itself. For more information, see the [Gemma formatting and system instructions document](https://ai.google.dev/gemma/docs/core/prompt-structure/).
  


------
> 💻 **Your task:**
>
> 1. **Implement conversation formatting**:
>    - **Special tokens**: The Gemma model uses `<start_of_turn>` and `<end_of_turn>`.
>    - **Role labels**: The Gemma model uses `user` and `model` roles.
>    - **Turn structure**: Adpapt the `format_io_as_turns()` function in the worked example to ensure it matches your conversation format requirements.
>
> 2. **Verify formatted output**:
>    - Write code to examine examples to verify:
>      - User turns are properly formatted with the prompt for classification.
>      - Model turns contain the expected classification responses.
>      - Special tokens are correctly positioned.
>      - The final data dictionary has the required "prompts" and "responses" keys.
>
> 3. **Customize for your domain**:
>    - Ensure the formatting aligns with how you want the model to behave during inference.
>

------

#### Worked example: Add special tokens


The code below transforms data that you prepared in the previous notebook into a conversational format suitable for fine-tuning a Gemma model.

The goal is to convert this into a training dataset containing user prompts and model responses with the summarization:

| Field | Purpose |
|-------|--------|
| **`input`** | The source text the model will summarize. |
| **`output`** | The concise summary the model should produce for this text. |

The cell also defines a function `extract_summary` which takes a raw model output including the prompt and returns only the generated summary. This will be useful for manual inspection of the summaries.

---



```python
# Special tokens for start/end of a turn.
SOT = "<start_of_turn>"
EOT = "<end_of_turn>"


def format_io_as_turns(
    input_text: str, output_text: str, sot: str = SOT, eot: str = EOT
) -> tuple[str, str]:
    """Format a single input/output pair for LoRA training.

    A single input/output pair is formatted for LoRA training by wrapping
    it with <start_of_turn> and <end_of_turn> markers.

    Args:
      input_text: The full user prompt text. This typically includes
        the instruction and the item content, e.g., "Summarize: ".
      output_text: The summary of the data point.
      sot: The token used to indicate the start of a turn.
        Defaults to "<start_of_turn>".
      eot: The token used to indicate the end of a turn.
        Defaults to "<end_of_turn>".

    Returns:
      formatted_q: The user turn string, e.g.
        "<start_of_turn>user\n{prompt}<end_of_turn>".
      formatted_a: the model turn string, e.g.
        "<start_of_turn>model\n{output}<end_of_turn>".
    """

    formatted_q = f"{sot}user\n{input_text}{eot}\n"
    formatted_a = f"{sot}model\n{output_text}{eot}"
    return formatted_q, formatted_a

def extract_summary(model_output: str) -> str:
    """Extracts the summary from a model output.

    Args:
      model_output: Raw output from the model.

    Returns:
      Summary extracted from the model output.
    """
    if "<start_of_turn>model" not in model_output:
        # Fallback: return model output directly
        return model_output.strip()
    response = model_output.split("<start_of_turn>model")[1].strip()
    if "<end_of_turn>" in response:
        response = response.split("<end_of_turn>")[0].strip()
    return response


# Load the dataset created by the instruction-based transformation step.
# Each line is a JSON object: {"input": "...", "output": "..."}

inputs = []  # Model prompts.
targets = []  # Expected responses.

for example in train_data:
    # Format each record into a user/model dialogue pair.
    q, a = format_io_as_turns(example["input"], example["output"])
    inputs.append(q)
    targets.append(a)

# Show the first set of input and output.
print(inputs[0])
print(targets[0])

# The Gemma3 Keras models require the data as dictionaries with keys
# "prompts" for the inputs and "responses" for the outputs.
data = {"prompts": inputs, "responses": targets}
```

#### Your implementation


In [ ]:
# Add your code to add special delimiter tokens here.


### Load Gemma using Keras
Once your data is ready, you now need to fine-tune your Gemma model using LoRA,  as you familiarized yourself with in Course 05 Fine-tune Your Model. You will train only a small number of additional parameters whilst keeping the base model frozen. You may find additional information in [Google's *Fine-tune Gemma in Keras using LoRA* documentation](https://ai.google.dev/gemma/docs/core/lora_tuning).

------
> 💡 **Tip: Curious about model size and performance tradeoffs?**
>
> You can experiment with the larger **Gemma3-4B** model to compare accuracy and output quality against the 1B version.  
> Keep in mind that Colab's default **T4 GPU** may not have enough memory to fine-tune the 4B model in full precision.  
> To make this feasible, apply the **optimization techniques** introduced in Course 07 Accelerate Your Model, such as mixed-precision inference, parameter-efficient fine-tuning, or low-bit quantisation, to reduce memory usage and stay within hardware limits.
>
------
More information about the Gemma3 models can be found in [Google's *Gemma3 model overview* document](https://ai.google.dev/gemma/docs/core).


#### Worked example: Load Gemma3



```python
# Load the Gemma3-1B Keras model.
model = keras_nlp.models.Gemma3CausalLM.from_preset("gemma3_1b")
model.summary()
```

#### Your implementation

In [ ]:
keras.utils.set_random_seed(812) # Replicate randomness in Keras.

# Add your code for loading a Gemma3 model here.


### Evaluate the base model's performance

Before fine-tuning, this is a good moment to **test the base (foundation) Gemma model** with a small set of example questions or prompts.
You can **reuse these same questions after fine-tuning** to see how LoRA adaptation changes the model's summarization outputs, giving you a direct, side-by-side comparison of performance.
For efficiency, **load the model only once** and keep it in memory while you carry out both the pre-fine-tuning tests and the later evaluation.
On Google Colab, where the default **T4 GPU has limited memory**, repeatedly loading different versions of the model can easily exceed available resources and trigger out-of-memory errors.

#### Worked example: Select examples for manual spot-checking

The following code selects fixed items from both the training and test sets to evaluate model performance. The selection is deterministic, ensuring the same items are used both before and after fine-tuning, which allows for direct comparison of how the model's outputs change. By comparing outputs on training examples (which the model has seen during fine-tuning) with test examples (which the model has not seen), you can assess whether the model is generalising well or overfitting to the training data. Strong performance on training items combined with poor performance on test items would indicate overfitting.

```python
# Select fixed items from training and test sets for evaluation.
n_samples = 3

# Select first n items from training set (items the model has seen).
train_samples = train_data[0:n_samples]

# Select first n items from test set (items the model has NOT seen).
test_samples = test_data[0:n_samples]

# Combine into a list of target items with source labels.
target_items = []

target_items.extend(train_samples)
target_items.extend(test_samples)

print(f"Selected {len(target_items)} items for evaluation.")

```

#### Your implementation


In [ ]:
# Add your code here.


Now, ask the base model to summarize each item using a simple instruction prompt. This will give you a baseline of how the model performs *before* any fine-tuning. For more information on running Gemma with Keras, see the [*Run Gemma with Keras* documentation](https://ai.google.dev/gemma/docs/core/keras_inference).

#### Worked example: Generate summaries and print them


```python
# Generate summaries and print them.

for item in target_items:
    prompt, _ = format_io_as_turns(item["input"], item["output"])

    # Print original text and the example summary from the dataset.
    print(f"Prompt: [{item['input']}")
    print(f"Example summary: {item['output']}")
    print("-" * 20)
    raw_model_output = model.generate(
        prompt, max_length=1024
    )
    summary = extract_summary(raw_model_output)
    print(summary)
    print("-" * 50)
```

#### Your implementation


In [ ]:
# Add your code for testing the base model here.


------
> ⚠️ **Understanding base model limitations**
>
> The base Gemma3-1B model may in some case produce empty outputs for example inputs. When using the non-instruction-tuned Gemma3 1B base model, prompts requesting structured tasks such as summarization may produce no output even when sufficient sequence length remains. This occurs because the base model has not learned instruction-response patterns and often assigns high probability to end-of-sequence tokens after long passages. As a result, generation may terminate immediately or produce only minimal continuation. This behavior is expected for base causal language models, particularly at smaller scales. In other cases, it may generate very incoherent responses rather than providing a proper summary.
>
> **Base models vs instruction-tuned models**:
>
> The Gemma3-1B model used here is a **base (pretrained) model**, not an **instruction-tuned** variant. Base models are trained on **next-token prediction**, learning to predict the most likely next word given the preceding context. They have not been further trained to understand and follow human instructions.
>
> In contrast, **instruction-tuned models** (such as `gemma3_instruct_1b`) undergo additional training on instruction-response pairs, teaching them to interpret prompts as commands and respond appropriately. When you ask a base model to 'summarize' something, it may simply continue generating text in a way that seems plausible given the input, rather than providing a proper summarization output.
>
> For more information about the model variants, see 🔗[Google's *Run Gemma content generation and inferences* documentation](https://ai.google.dev/gemma/docs/run).
>
> **Why small models struggle more**:
>
> With only 1 billion parameters, the Gemma3-1B model has limited capacity to capture complex patterns and implicit task instructions from context alone. Larger models can sometimes infer task requirements from prompt phrasing, but smaller base models are more prone to producing off-topic or hallucinated responses, generating text that sounds plausible but is factually incorrect or irrelevant to the task, going into producing gibberish, or just remaining silent.
>
> **Why use a base model for fine-tuning**:
>
> Despite these limitations, base models are often preferred for fine-tuning. Because they have not been optimized for a specific instruction-following format, they offer a more neutral starting point. This allows LoRA fine-tuning to more effectively reshape the model's behavior toward your specific summarization task, without having to override patterns learned during instruction tuning.
>
> By comparing the base model's outputs before and after fine-tuning, you will see a clear demonstration of how LoRA adaptation transforms an unreliable base model into a capable summarizer.
------


## Activity 3: Activate and run LoRA



------
> 💻 **Your task:**
>
> Set up your hyperparameters and fine-tune your Gemma model using LoRA:
>
> 1. **Experiment with LoRA rank**: Start with `rank=6` or `rank=8` for better quality, or try `rank=4` for faster training:
>    ```python
>    lora_rank = 6
>    model.backbone.enable_lora(rank=lora_rank)
>    ```
>
> 2. **Adjust training parameters**:
>    - **Learning rate**: Start with `5e-5`. You can experiment with other rates such as `1e-5` or `2e-5`.
>    - **Number of epochs**: Begin with 2 epochs for quick iteration. Increase if needed, but monitor for overfitting.
>    - **Batch size**: Set to `1` due to limited memory on T4 GPUs. If you have access to a GPU with more memory, you can raise this to speed up training.
>    - **Sequence length**: Check how long the summaries are and set a value based on that.
>
> 3. **Monitor training progress**:
>    - Watch the loss decrease over epochs.
>    - Check sparse categorical accuracy improvement.
>    - Stop early if the model starts overfitting.
>
> 4. **Iterate as needed**: If results are not satisfactory:
>    - Try different hyperparameters.
>    - Increase/decrease epochs.
>    - Adjust the LoRA rank.
>
------


#### Worked example: Set hyperparameters.


```python
# Set the LoRA rank.
lora_rank = 16
# Set the learning rate.
learning_rate = 1e-4
# Determine the number of epochs.
num_epochs = 2
# Set the batch size.
batch_size = 1
# Set the maximum length.
sequence_length = 530

# Activate LoRA.
model.backbone.enable_lora(rank=lora_rank)

# Check learning rate.
print(model.optimizer.learning_rate)
```

#### Your implementation


In [ ]:
# Add your code for setting hyperparameters here.


-----
> **ℹ️ Info: Understanding LoRA hyperparameters**
>
> The following parameters control how LoRA fine-tuning adapts the model to your summarization task:
>
> **`lora_rank`** (integer, e.g., 4, 8, 12, 16):
> The rank of the low-rank matrices added to the model's attention layers. Higher ranks capture more complex patterns but increase memory usage and training time. For small datasets or simple summarization tasks, `rank=8` is often sufficient. For larger datasets or tasks requiring nuanced understanding, try `rank=12` or higher.
>
> **`learning_rate`** (float, e.g., 1e-5, 5e-5):
> Controls how much the model weights are adjusted during each training step. Higher rates learn faster but risk overshooting optimal values. Lower rates are more stable but train slower. Start with `2e-5` or `5e-5` for LoRA fine-tuning. If the model fails to learn (high loss), try increasing slightly. If training is unstable (loss fluctuates), reduce the rate.
>
> **`num_epochs`** (integer, e.g., 2, 5, 10):
> The number of complete passes through the training dataset. More epochs allow the model to learn more thoroughly, but too many can cause overfitting to the training data rather than learning generalizable patterns. Start with 2-3 epochs for small datasets and increase if validation performance is still improving (monitor the `loss` and `sparse_categorical_accuracy` values). Also monitor the gap between seen and unseen data performance later on during the evaluation of the model.
>
> **`batch_size`** (integer, e.g., 1, 4, 8):
> The number of training examples processed together before updating weights. Larger batches provide more stable gradient estimates but require more GPU memory. On T4 GPUs with limited memory, `batch_size=1` is often necessary for large models like Gemma. If memory permits, increasing batch size can speed up training.
>
> **`sequence_length`** (integer, e.g., 256, 384, 512):
> The maximum number of tokens the model processes per input. Must be long enough to contain your full prompts and expected responses. Longer sequences require more memory. Analyze your training data to determine appropriate length and set it slightly above your longest expected input/output combination. The default value of 530 is close to the upper limit for standard LoRA training of a Gemma3 1B model on a T4 GPU with 16 GB of RAM.
>
-----


### Fine-tuning
It is now time to perform fine-tuning with LoRA. You can leave the hyperparameters as default or change them. You may wish to reduce the number of epochs. In the first run watch the loss and the sparse categorical accuracy for each epoch and adjust accordingly to achieve the desired accuracy without overfitting the model.

The time required to complete LoRA fine-tuning depends on several factors, including the size of the dataset, the length of the prompts, and hyperparameters such as the number of epochs and LoRA rank. Depending on these variables, the process can take anywhere from a few minutes to several hours. For the default example dataset, the estimated fine-tuning time is about 15 minutes on a T4 GPU.

#### Worked example: Run fine-tuning


```python
model.optimizer.learning_rate = learning_rate
model.preprocessor.sequence_length = sequence_length

# Using a for loop provides better control over each iteration if you want to
# add custom logic, but otherwise identical to setting epochs=num_epochs.
for i in jnp.arange(num_epochs):
    print("\n\nEpoch:" + str(i + 1) + "\n")
    model.fit(data, epochs=1, batch_size=batch_size, verbose=1)
    # Add code to print model generations for one or two prompts after every
    # epoch here.
```

#### Your implementation

In [ ]:
# Add your code to fine-tune the model.


Ideally, you now have a fine-tuned Gemma model trained with LoRA for your summarization task. The next step is to evaluate how well the summarization model performs by testing it on real examples and conducting systematic evaluation.

## Activity 4: Model evaluation



------
> 💻 **Your task:**
>
> Evaluate your fine-tuned summarization model through manual spot checks and qualitative assessment. You will assess performance on both **seen data** (training examples from `df_train`) and **unseen data** (test examples from `df_test`) to understand how well your model generalizes.
>
> 1. **Compare base model vs. fine-tuned model**:
>    - The `target_items` list contains 3 items from the training set and 3 from the test set, selected deterministically for consistent comparison.
>    - Run the evaluation code on the same items you tested with the base model earlier.
>    - Observe the difference in model behavior:
>      - Does the fine-tuned model now produce coherent, concise summaries?
>      - How does the output compare to the base model's responses?
>
> 2 **Run spot checks using the helper functions**:
>    - `run_spot_check(data, sample_size)` generates summaries for a sample of items.
>    - `extract_summary()` cleans the model output and places results in `generated_summary`.
>    - `display_spot_check(df_spot, title)` displays results in a readable wrapped format.
>    - Run spot checks on both `df_train` (seen) and `df_test` (unseen) data.
>
> 3. **Customize evaluation parameters**:
>    - **`sample_size`**: Set number of examples to evaluate (start with 5-10, increase for thorough testing).
>    - **`random_state`**: Change random seed (default: 42) for different sample selection, or remove for true randomness.
>    - **`summarization_instruction`**: Modify to match your specific task and domain (e.g., desired summary length or focus).
>    - **Column names**: Update references like `"expanded"`, `"summary"`, etc. to match your data structure.
>
> 4. **Analyze length ratios**:
>    - Run the length ratio analysis code to compare compression behavior between seen and unseen data.
>    - Similar ratios suggest the model has learned a consistent summarization style.
>    - Compare results to your original training data to assess how well the model matches your target summary length.
>
> 5. **Generate summaries for all test data**:
>    - Run the code that processes all items in `df_test`.
>    - Results are stored in `df_results` with the `generated_summary` column.
>    - Review the sample table comparing reference and generated summaries.
>
> 6. **Export results to CSV**:
>    - Run the export code to save results to `summarization_results.csv`.
>    - The file includes source text, reference summary, and generated summary.
>
> 7. **Assess summary quality**:
>    - **Content coverage**: Does the summary capture the key points?
>    - **Faithfulness**: Is the summary factually accurate to the source?
>    - **Conciseness**: Is the summary appropriately shorter than the original?
>    - **Fluency**: Is the summary grammatically correct and readable?
>    - **Generalization**: Is there a quality difference between seen and unseen data?
>
> 8. **Document your findings**: Record what works well and what needs improvement.
>
------

### Comparing base model vs fine-tuned model outputs

Now that the model has been fine-tuned with LoRA, you can test it using the **same texts** you used earlier to evaluate the base model. This allows for a direct, side-by-side comparison of how fine-tuning has affected the model's summarization behavior.

In the following cell, implement code to generate summaries with your fine-tuned model for the set of texts that you used to evaluate your base model. Compare the outputs with what you observed earlier from the base model.


#### Worked example: Spot-check summaries

```python
# Generate summaries using the fine-tuned model and print them.

for item in target_items:
    prompt, _ = format_io_as_turns(item["input"], item["output"])

    # Print original text and the example summary from the dataset.
    print(f"Prompt: [{item['input']}")
    print(f"Example summary: {item['output']}")
    print("-" * 20)
    raw_model_output = model.generate(
        prompt, max_length=1024
    )
    summary = extract_summary(raw_model_output)
    print(summary)
    print("-" * 50)
```

#### Your implementation


In [ ]:
# Add your code for generating summaries here.


#### What to observe

You likely observed the following differences:

- **Base model (before fine-tuning)**: This model likely produced inconsistent, off-topic, or hallucinated responses because it was trained only on next-token prediction and did not understand the summarization task.

- **Fine-tuned model (after LoRA)**: This model should ideally produce more coherent, concise summaries that capture the key information from the source texts, demonstrating how LoRA adaptation has taught the model to follow the summarization instruction.


#### Generate summaries for all test data

Having validated the model's performance on a small sample, you can now generate summaries for the entire test set. This creates a comprehensive evaluation dataset that can be reviewed manually, used for further analysis, or exported for external review. The code below processes all items in `test_df`, generating summaries and extracting the clean output text into a new column. This step typically takes about 5 minutes to complete for the example dataset, as approximately 140 items need to be summarized.

#### Worked example: Generate summaries for all test data.


```python
# Generate summaries for all test data.

# Create a copy of test data for results.
df_results = test_df.copy()

# Generate prompts for all test items.
prompts = []
for t in df_results["input"]:
    formatted_text, _  = format_io_as_turns(str(t), "")
    prompts.append(formatted_text)

df_results["prompt"] = prompts

# Generate model outputs (this may take a few minutes).
print(f"Generating summaries for {len(df_results)} test items.")
outputs = []
for prompt in tqdm(df_results["prompt"]):
    output = model.generate(prompt, max_length=1024)
    outputs.append(output)

df_results["model_output"] = outputs

# Extract clean summaries from model outputs.
generated_summaries = []
for output in df_results["model_output"]:
    summary = extract_summary(output)
    generated_summaries.append(summary)

df_results["generated_summary"] = generated_summaries

print(f"Generation complete. {len(df_results)} summaries generated.")
```

#### Your implementation


In [ ]:
# Add your code for  here.


The results DataFrame now contains all test items with their generated summaries. The following code displays a sample of the results, showing the reference summary alongside the model's generated summary for comparison.

#### Worked example: Display sample results comparing reference and generated summaries


```python
# Display sample results comparing reference and generated summaries.
sample_cols = ["input", "output", "generated_summary"]
display(df_results[sample_cols].head(10))
```

#### Your implementation


In [ ]:
# Add your code here.


To facilitate external review or further analysis, you can export the results to a CSV file. The exported file includes the original text, reference summary, and generated summary for each test item.

#### Worked example: Save summaries in a CSV file


```python
# Define columns to export.
export_cols = [
    "input",
    "output",
    "generated_summary"
]

# Export to CSV and save to Google Drive.
output_filename = "/content/drive/MyDrive/summarization_results.csv"
df_results[export_cols].to_csv(output_filename, index=False, encoding="utf-8")

print(f"Results saved to '{output_filename}'")
print(f"Total rows exported: {len(df_results)}")
```

#### Your implementation


In [ ]:
# Add your code here.


#### Evaluating model summaries: spot-checking guide

Once you've generated all the summaries, you can evaluate them through systematic spot checks. This involves evaluating your model's performance by comparing the generated summaries against the source text and reference summaries.

#### Step-by-step evaluation process for worked example

To provide some guidance on how to go about spot-checking examples, consider this step-by-step evaluation process for the worked example. Adapt this to your specific summarization task.

**1. Compare with the source text**
Read the original `input` to understand what information should be captured.

**2. Assess summary quality**
Some useful dimensions to consider are:
- ✅ **Content coverage**: Does it capture the key points from the source?
- ✅ **Faithfulness**: Is it factually accurate to the source text?
- ✅ **Conciseness**: Is it appropriately shorter than the original?
- ✅ **Fluency**: Is it grammatically correct and readable?

#### Example

| description | summary | model_output | Evaluation |
|-------------|---------|--------------|------------|
| Jollof rice is a popular... | Jollof is a West African rice dish... | ...Jollof rice is a beloved West African one-pot dish... | ✅ GOOD - Captures key info, faithful, concise |
| The baobab tree grows... | Baobabs are iconic African trees... | ...Trees are plants that grow in Africa... | ⚠️ WEAK - Too vague, missing key details |

#### What to look for during spot-checking

**✅ Good signs:**
- Generated summaries capture the main points from source texts.
- Summaries are factually accurate and do not introduce false information.
- Summaries are appropriately concise relative to the source length.
- Language is fluent and natural-sounding.

**⚠️ Warning signs:**
- Key information from the source is missing.
- Summaries contain hallucinated facts not in the source.
- Summaries are too long or merely copy the source text.
- Awkward phrasing or grammatical errors.
- Inconsistent summary style or length across examples.

#### Systematic evaluation tips

1. **Check faithfulness**: Verify that no false information has been introduced.
2. **Assess coverage**: Note which types of information are consistently captured or missed.
3. **Evaluate consistency**: Look for consistent summary style and quality across examples.

#### Adapting for your dataset

**Important**: Remember to adjust the evaluation process based on your specific summarization task:
- Consider your target summary length and style.
- Adjust evaluation criteria based on your domain (e.g., technical accuracy for scientific summaries).
- Note whether your summaries should be extractive (using source phrases) or abstractive (rephrasing).
- Consider your domain-specific requirements for what information must be preserved.

This manual evaluation process helps you understand your model's strengths and weaknesses before deploying it in practical applications.


#### Length ratio analysis

Beyond reading individual summaries, a simple quantitative check is to examine the **compression ratio**: how much shorter are the generated summaries compared to their source texts? A well-functioning summarizer should produce outputs that are substantially shorter than the inputs while still capturing the key information.

The length ratio is calculated as:

$$\text{Length ratio} = \frac{\text{Summary word count}}{\text{Source word count}}$$

A ratio of 0.10 means the summary is 10% of the source length. Lower ratios indicate more aggressive compression.

#### Worked example: Compute length ratios


```python
def compute_length_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute length ratios between generated summaries and source texts.

    Args:
      df: DataFrame with 'input' and 'generated_summary' columns.

    Returns:
      DataFrame with added word count and ratio columns.
    """
    result = df.copy()

    # Compute word counts.
    result["source_words"] = result["input"].astype(str).str.count(r"\S+")
    result["summary_words"] = result["generated_summary"].astype(str).str.count(r"\S+")

    # Avoid divide-by-zero and set ratio to NaN when source_words is 0.
    result["length_ratio"] = result["summary_words"] / result["source_words"].replace(0, pd.NA)

    return result


def display_length_stats(df: pd.DataFrame, label: str):
    """
    Display summary statistics for source and summary text lengths.

    Args:
      df: DataFrame containing 'source_words', 'summary_words', and
        'length_ratio' columns.
      label: Descriptive label printed before the statistics (e.g.,
        dataset name or split).
    """
    print(f"\n{label}")
    print(f"  Source text:  {df['source_words'].mean():.1f} words (avg)")
    print(f"  Summary:      {df['summary_words'].mean():.1f} words (avg)")
    print(f"  Length ratio: {df['length_ratio'].mean():.2%} (avg)")
    print(f"  Ratio range:  {df['length_ratio'].min():.2%} - "
          f"{df['length_ratio'].max():.2%}")


# Compute length ratios for unseen examples.
df_unseen_ratios = compute_length_ratios(df_results)

# Display statistics.
print("=" * 50)
print("Length ratio analysis")
print("=" * 50)

display_length_stats(df_unseen_ratios, "Unseen data (test set):")

```

#### Your implementation


In [ ]:
# Add your code for computing length ratios here.


The length ratio analysis provides a quick sense check on the model's compression behavior and gives you an idea of how much the model compresses the original texts.

Note, however, that length ratio alone does not measure summary quality. A very short summary might miss important information, while a longer one might include unnecessary details. Use this metric alongside the qualitative spot checks to build a complete picture of model performance.

-----
> **💡 Tip: Using length ratios to guide training data design**
>
> You can run the same length ratio analysis on your original training dataset to establish a baseline. Comparing the fine-tuned model's output ratios to the training data ratios reveals how closely the model has learned to match your target summary style.
>
> Fine-tuned models tend to produce summaries of similar length to what they were trained on. Longer reference summaries in the training data will result in longer generated summaries, and vice versa. This behavior can be leveraged deliberately:
>
> - **Instruction-based training**: Create training examples with explicit instructions such as 'Summarize briefly in 1-2 sentences' paired with short summaries, and 'Provide a detailed summary' paired with longer summaries. The model can learn to vary its output length based on the instruction.
>
> - **Task-specific datasets**: If your application requires a specific compression ratio (e.g., executive summaries at 5% of source length), curate training examples that consistently demonstrate this ratio.
>
> This approach gives you fine-grained control over summary length without needing to adjust generation parameters at inference time.
>
-----

## Activity 5: Iteration

One of the key parts of building any machine learning system is experimenting with different hyperparameters, datasets, models, or other variations of the pipeline that you just implemented. Even the most experienced machine learning researchers and engineers rarely get all of these things right without systematic experiments.

If you identified any systematic issues or are otherwise not yet satisfied with the performance of your summarization system, think how you may be able to improve your system. You can then go back to individual steps, make changes and repeat the training and evaluation of your model to determine whether the changes are effective.

To do these iterations most systematically, it is best if you do not change too many things at once, so that you learn which components affect your summarization system the most. Furthermore, make sure to keep a log of your experiments somehwhere in which you detail the settings of each training run, so that you know at the end what worked best. That way, you can progressivly optimize your machine learning model until you are satisfied with its performance.

## Activity 6: Reflection



------
> 💻 **Your task:**
>
> **Technical and project reflection:**
>
> Document your learning and plan improvements.
>
> - **What worked well:** Reflect on successful aspects of your summarization model
>
> - **Challenges encountered:** What difficulties did you face and how did you solve them?
>
> - **Data quality observations:** How was the quality of your training data? What would you do differently?
>
> - **Model performance:** How well did your summarization model perform? What are its strengths and weaknesses?
>
> - **Summarization-specific insights:** What did you learn about text summarization? Which types of content were hardest to summarize effectively?
> ------
> **Learning journey reflection:**
>
> Take a few minutes to think about your own learning journey in this course. Use these prompts to guide a short written reflection.
>
>  - **What aspects of the course helped you understand large language models and LoRA fine-tuning most effectively?** Which study habits or tools did you find especially valuable for your progress?
>
>  - **How did your thinking about machine-learning workflows change as you moved from data collection to fine-tuning and evaluation?** What surprised you or challenged your assumptions?
>
>  - **Looking beyond this specific project, how will you apply these skills and insights when approaching new ML tasks or unfamiliar datasets in the future?**
>
>  - **What attitudes, ethics, or professional values do you believe are essential for a responsible ML practitioner?** How will you cultivate these qualities as you continue to develop as a machine-learning engineer?
------

No example solution is provided for the reflection activity. This exercise is designed to be completed independently by you, drawing on your own experiences, challenges, and insights from working through this Capstone project. Your reflections are personal and unique to your learning journey.

### Key findings

_[Summarize your main observations and results from this project]_

-
-
-


### Challenges encountered

_[Describe technical, conceptual, or resource-related challenges]_

-
-
-


### Future improvements

_[What would you do to enhance this system?]_

-
-
-


### Course connections

_[Which course concepts were most important for this project?]_


## Summary

Congratulations, you have completed the summarization capstone project.

This capstone demonstrated a complete machine learning workflow from conception to evaluation, highlighting the importance of careful data curation, parameter-efficient training methods, and evaluation practices in building reliable AI systems.